In [1]:
!pip install imdbpy pandas

In [2]:
import pandas as pd

in_scope_ids = set()
foreign_born_ids = set()

# Expanded exclusion list based on the languages/regions you identified
# JA -> JP (Japan), IT (Italy), ES (Spain), FR (France), DE (Germany), PT (Portugal)
# Also including US/GB/CA/AU to keep Hollywood out.
exclude_regions = {
    'US', 'GB', 'CA', 'AU',  # English hubs
    'JP', 'IT', 'ES', 'FR', 'DE', 'PT',  # The regions you listed
    'CN', 'KR', 'RU'  # Other major international film hubs
}

print("Step 1: Identifying Indian IDs and filtering out foreign originals...")

for chunk in pd.read_csv('title.akas.tsv.gz', sep='\t', compression='gzip',
                         chunksize=500000, low_memory=False,
                         usecols=['titleId', 'region', 'isOriginalTitle']):

    # 1. Harvest anything with an 'IN' tag
    in_matches = chunk[chunk['region'] == 'IN']['titleId'].unique()
    in_scope_ids.update(in_matches)

    # 2. Identify movies where the 'Original' version belongs to an excluded region
    foreign_matches = chunk[(chunk['region'].isin(exclude_regions)) & (chunk['isOriginalTitle'] == 1)]['titleId'].unique()
    foreign_born_ids.update(foreign_matches)

# Final Clean List: (Released in India) MINUS (Originally from a foreign hub)
indian_ids = in_scope_ids - foreign_born_ids

print(f"Success! Filtered down to {len(indian_ids)} strictly Indian movie IDs.")


Step 1: Identifying Indian IDs and filtering out foreign originals...
Success! Filtered down to 5425542 strictly Indian movie IDs.


In [3]:
movies_list = []
print("Step 2: Cross-checking with Basics to remove foreign films...")

# Using title.basics.tsv.gz
for chunk in pd.read_csv('title.basics.tsv.gz', sep='\t', compression='gzip',
                         chunksize=500000, low_memory=False,
                         usecols=['tconst', 'titleType', 'primaryTitle', 'originalTitle', 'startYear', 'genres']):

    # Filter for movies in our 'IN' list
    in_scope = chunk[(chunk['tconst'].isin(indian_ids)) & (chunk['titleType'] == 'movie')].copy()

    # DOUBLE CROSS-CHECK LOGIC:
    # Rule A: Keep if primaryTitle != originalTitle (Transliterated Indian films)
    # Rule B: Keep if genres contains 'Musical' (Classic Indian Cinema trait)
    rule_a = in_scope['primaryTitle'] != in_scope['originalTitle']
    rule_b = in_scope['genres'].str.contains('Musical', na=False)

    matches = in_scope[rule_a | rule_b]
    movies_list.append(matches)

final_movies = pd.concat(movies_list).drop_duplicates('tconst')
final_movies.replace('\\N', '').to_csv('movies.csv', index=False)
print(f"movies.csv created with {len(final_movies)} verified Indian films.")


Step 2: Cross-checking with Basics to remove foreign films...
movies.csv created with 19336 verified Indian films.


In [7]:
roles_list = []
target_tconsts = set(final_movies['tconst'])
print("Step 3: Extracting roles for verified movies...")

# Using title.principals.tsv.gz
for chunk in pd.read_csv('title.principals.tsv.gz', sep='\t', compression='gzip',
                         chunksize=500000, low_memory=False,
                         usecols=['tconst', 'nconst', 'category']):
    matches = chunk[chunk['tconst'].isin(target_tconsts)]
    roles_list.append(matches)

roles_df = pd.concat(roles_list)
roles_df.to_csv('roles.csv', index=False)
print(f"roles.csv created with {len(roles_df)} entries.")

Step 3: Extracting roles for verified movies...
roles.csv created with 346079 entries.


In [8]:
people_list = []
target_nconsts = set(roles_df['nconst'])
print("Step 4: Fetching names and birth years...")

# Using name.basics.tsv.gz
for chunk in pd.read_csv('name.basics.tsv.gz', sep='\t', compression='gzip',
                         chunksize=500000, low_memory=False,
                         usecols=['nconst', 'primaryName', 'birthYear']):
    matches = chunk[chunk['nconst'].isin(target_nconsts)]
    people_list.append(matches)

final_people = pd.concat(people_list).drop_duplicates('nconst')
final_people.replace('\\N', '').to_csv('people.csv', index=False)
print("people.csv created. All files are ready!")



Step 4: Fetching names and birth years...
people.csv created. All files are ready!
